In [1]:
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import torch
import scipy.spatial
import numpy as np
from datasets import load_dataset
import faiss

/Users/bahloulia/Downloads/agentic_software/RAG-Applications/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Way 1 — FAISS (in-memory similarity search)

Baseline approach: embed the Paris-only reviews with `nomic-embed-text-v1.5`, build an in-process `faiss.IndexFlatIP` index (cosine similarity via L2-normalized vectors), retrieve the top-k most similar reviews, and generate a cited answer from them with Claude.

No persistence, no metadata filtering beyond the pre-filtered `df_paris` slice, no server — everything lives in this Python process and is rebuilt from scratch each run.

In [2]:
dataset = load_dataset("traversaal-ai-hackathon/hotel_datasets")
df=pd.DataFrame(dataset['train'])
 
df_paris = df.loc[df.locality=='Paris']
df_paris.drop_duplicates()
reviews = df_paris['review_text'].tolist()

def get_embeddings(data, model):
    embeddings = model.encode(data, show_progress_bar=False).astype('float32')
    return embeddings
 
def search_faiss_index(query_embedding, faiss_index, k=5):
    query_embedding_normalized = query_embedding / np.linalg.norm(query_embedding)
    distances, indices = faiss_index.search(query_embedding_normalized, k)
    return distances, indices
 
def create_faiss_index(embeddings):
    embeddings_normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    index = faiss.IndexFlatIP(embeddings_normalized.shape[1])
    index.add(embeddings_normalized)
    return index


reviews = df_paris['review_text'].tolist()
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
if torch.cuda.is_available():
    model = model.to('cuda')
 
review_embeddings = get_embeddings(reviews, model)
faiss_index = create_faiss_index(review_embeddings)

<All keys matched successfully>


In [3]:
import json
 
def search_hotels_by_query(query, model, faiss_index, df, k=25):
    query_embedding = get_embeddings([query], model)
    distances, indices = search_faiss_index(query_embedding, faiss_index, k=k)
 
    results = []
    for i, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
        results.append({
            "rank": i,
            "hotel_name": df.iloc[idx]['hotel_name'],
            "review_text": df.iloc[idx]['review_text'],
            "cosine_similarity": float(distance) # Convert float32 to standard float
        })
 
 
    return results
 
query = "Hotel with a view of the Eiffel tower."
json_output = search_hotels_by_query(query, model, faiss_index, df_paris, k=25)
print(json_output)

[{'rank': 1, 'hotel_name': 'Pullman Paris Eiffel Tower Hotel', 'review_text': 'Excellent service. Stunning view of the Eiffel towere from our balcony, The room is gorgeous, comfortable and spacious. Definitely will be recommending this hotel to family and friends. If you’re looking for a hotel that has everything you need in Paris, luxury and view, this is the one.', 'cosine_similarity': 0.8226630091667175}, {'rank': 2, 'hotel_name': 'Pullman Paris Eiffel Tower Hotel', 'review_text': 'If you stay at this hotel it is for the amazing views of the Eiffel Tower and for the pictures.  The photos and memories of being on the balcony looking at the Eiffel Tower in all its splendor cannot be denied.   When we arrived we were impressed with proximity of hotel to the Eiffel Tower. Definitely within walking distance, 7 minutes, we could even see our hotel when we climbed up the tower later that day . The view itself is spectacular!! We were upgraded to a suite top floor (9th floor) with a balcony

In [4]:
from anthropic import Anthropic

# Reads the ANTHROPIC_API_KEY environment variable automatically.
anthropic_client = Anthropic()

In [5]:
def generate_answer(query):
    json_output = search_hotels_by_query(query, model, faiss_index, df_paris, k=25)
    prompt = f"""
    Based on the following query from a user, please generate a small answer
    focusing on the original query and the response given. The answer should be paragraphs.
    Remove the special characters and (/n), make the output clean and long.
    Please cite source for each part as [1][2].
    Just start with the answer, no need to give any salutations.
 
    ###########
    query:
    "{query}"
 
    ########
 
    context:
    "{json_output}"
    #####
 
    Return in Markdown format.
    """
    output_text = ""
    with anthropic_client.messages.stream(
        model="claude-haiku-4-5",
        max_tokens=2048,
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        for text in stream.text_stream:
            output_text += text  # Append new content to the full output
            print(text, end="")  # Print each chunk live as it's received

    return output_text, json_output

In [6]:
query = "Hotels with a view of the Eiffel Tower"
response,sources = generate_answer(query)
print(response)

# Hotels with a View of the Eiffel Tower

Paris offers several excellent hotel options for travelers seeking accommodations with stunning views of the iconic Eiffel Tower. These establishments range from luxury properties to more affordable alternatives, each providing unique experiences for visitors wanting to maximize their Parisian stay.

The Pullman Paris Eiffel Tower Hotel stands out as one of the premier choices for guests seeking exceptional Eiffel Tower views [1]. Reviewers consistently praise the hotel's location, which is within walking distance of the tower, approximately seven minutes on foot [1]. Guests staying at this property enjoy gorgeous, comfortable, and spacious rooms with balconies offering stunning panoramic views of the Eiffel Tower [1]. The hotel provides complementary breakfast service and excellent amenities, making it a comprehensive choice for those seeking luxury and memorable vistas [1].

For budget-conscious travelers, Citadines Tour Eiffel Paris presents

The generated response showcases several key advantages over plain search results. First, it provides contextual analysis by comparing different hotels and their specific view offerings, from the Pullman's "exceptional view" with "stunning balcony vistas" to Citadines' more modest kitchen-area views. Second, it synthesizes information from multiple data points—room types, guest reviews, location details, and amenities—into coherent paragraphs that tell a complete story about each property. Third, it maintains source attribution through numbered citations, allowing users to trace specific claims back to their origins while presenting information in a natural, conversational format that's far more digestible than raw search results

## Way 2 — Qdrant (vector database)

Same embedding model, but indexed into a Qdrant collection instead of FAISS: the full (unfiltered) dataset is embedded and upserted with `hotel_name`, `review_text`, and `locality` stored as payload, so retrieval can filter by city at query time (`search_qdrant_with_filter`) instead of needing a separate pre-filtered DataFrame per city.

Qdrant runs in `:memory:` mode here for the notebook, but the same client code points at a real Qdrant server (`docker run -p 6333:6333 qdrant/qdrant`) to get persistence, concurrent access, and incremental upserts — the production-service properties FAISS doesn't provide.

In [13]:

from qdrant_client import QdrantClient, models
 
client = QdrantClient(":memory:")
text_embeddings_size = 768
dataset = load_dataset("traversaal-ai-hackathon/hotel_datasets")
df_all = pd.DataFrame(dataset['train'])
df_all.drop_duplicates()
reviews_df = df_all.dropna(subset=['review_text'])
reviews = reviews_df['review_text'].tolist()
 
review_embeddings = get_embeddings(reviews, model)

In [14]:
def create_qdrant_collection(client, collection_name, vector_size):
    try:
        if client.collection_exists(collection_name):
            client.delete_collection(collection_name=collection_name)
            print(f"Collection '{collection_name}' deleted successfully.")

        client.create_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(
                size=vector_size,
                distance=models.Distance.COSINE
            ),
        )

        print(f"Collection '{collection_name}' created successfully.")

    except Exception as e:
        print(f"An error occurred while setting up the collection: {e}")

In [15]:
collection_name = "hotel_reviews"
create_qdrant_collection(client, collection_name, text_embeddings_size)

Collection 'hotel_reviews' created successfully.


In [16]:
def upload_reviews_to_qdrant(client, collection_name, reviews_df, review_embeddings):
    points_to_upload = [
        models.PointStruct(
            id=i,  # Use the index as the unique ID
            vector=review_embeddings[i].tolist(),
            payload={
                "hotel_name": reviews_df.iloc[i]['hotel_name'],
                "review_text": reviews_df.iloc[i]['review_text'],
                "locality": reviews_df.iloc[i]['locality'],  # Include locality for filtering
            },
        )
        for i in range(len(review_embeddings))
    ]

    return client.upsert(
        collection_name=collection_name,
        wait=True,
        points=points_to_upload,
    )

In [17]:
upload_reviews_to_qdrant(client, collection_name, reviews_df, review_embeddings)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [18]:
def search_qdrant_with_filter(query, model, client, city=None, k=10):
    query_embedding = get_embeddings([query], model)[0]

    query_filter = None
    if city:
        query_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key="locality",
                    match=models.MatchValue(value=city)
                )
            ]
        )

    text_hits = client.query_points(
        collection_name="hotel_reviews",
        query=query_embedding,
        query_filter=query_filter,
        limit=k,
        with_payload=True,
    ).points

    return text_hits

In [19]:
def generate_answer_qdrant(query, city=None):
    qdrant_results = search_qdrant_with_filter(query, model, client, city=city, k=25)

    context_string = ""
    for i, result in enumerate(qdrant_results):
        context_string += f"Source {i+1}:\n"
        context_string += f"Hotel: {result.payload.get('hotel_name', 'N/A')}\n"
        context_string += f"Review: {result.payload.get('review_text', 'N/A')}\n"
        context_string += f"Locality: {result.payload.get('locality', 'N/A')}\n"
        context_string += f"Similarity Score: {result.score:.4f}\n\n"

    prompt = f"""
    Based on the following query from a user and the provided context from hotel reviews,
    please generate a concise answer summarizing the relevant information.
    Focus on addressing the user's query using details found in the reviews.
    Cite the sources using numerical references like [1], [2], etc., corresponding to the "Source #" in the context.
    Format the output as a few paragraphs.
    Remove any special characters like (/n) and ensure the output is clean.
    Begin directly with the answer.
 
    ###########
    query:
    "{query}"
 
    ########
 
    context:
    "{context_string}"
    #####
 
    Return in Markdown format.
    """

    output_text = ""
    with anthropic_client.messages.stream(
        model="claude-haiku-4-5",
        max_tokens=2048,
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        for text in stream.text_stream:
            output_text += text  # Append new content to the full output
            print(text, end="")  # Print each chunk live as it's received

    return output_text, qdrant_results

In [20]:
query= "Amazing hotel close to everything"
city_filter = "Istanbul"
response_paris, sources_paris = generate_answer_qdrant(query, city=city_filter)
 
print("\n--- Generated Answer (City Filter) ---")
 
print("\n--- Sources Used (City Filter) ---")
for i, result in enumerate(sources_paris):
    print(f"Source {i+1}: Hotel: {result.payload.get('hotel_name', 'N/A')}, Locality: {result.payload.get('locality', 'N/A')}, Score: {result.score:.4f}")
 
print("\n" + "="*50 + "\n") # Separator

# Amazing Hotels Close to Everything in Istanbul

Istanbul offers several exceptional hotels perfectly positioned near major attractions and amenities. These properties consistently deliver on the promise of convenience combined with outstanding service and comfort.

Several hotels stand out for their unbeatable locations. The Skalion Hotel & Spa is praised for having every main attraction—including the Grand Bazaar, Hagia Sophia, and Blue Mosque—within walking distance, with hundreds of markets, restaurants, pharmacies, and currency exchanges literally attached to the hotel, eliminating the need for taxis [2]. Similarly, Hotel Yasmak Sultan offers a location described as "perfect," with everything within walking distance, including easy access to the Galata crossing by a beautiful 10-15 minute walk [19]. The Mula Hotel provides a great location just 5-10 minutes' walk from stunning landmarks like Ayasofya and Sultan Ahmed historical mosques [13].

Beyond proximity to attractions, thes